# 02 - Baseline Training Loop with Space-Domain & Photometric Augmentations

This notebook executes the **YOLOv8-Nano baseline training loop** for space debris and spacecraft detection.

### Key Features & Synthetic-to-Real Gap Mitigation:
1. **Photometric Augmentation Handoff (`src.data.augment`):** Synthetic renders (SPARK / SwissCube) are artificially clean. Before training, we inject random **Gaussian sensor noise**, **Gaussian blur**, and **exposure/contrast shifts** into training images so the detector learns to generalize to real telescope imagery.
2. **Space-Domain Orbital Augmentations:** Spacecraft and debris tumble unpredictably in zero gravity. We configure **full 360° rotation invariance (`degrees=180.0`)**, **vertical/horizontal flips (`flipud=0.5`, `fliplr=0.5`)**, and **random erasing (`erasing=0.4`)** to simulate solar panel and antenna occlusions.
3. **GPU Acceleration & RAM Caching:** Uses CUDA (`NVIDIA RTX 2050`) with dataset RAM caching (`cache=True`) to eliminate disk I/O latency.

In [1]:
%pip install ultralytics albumentations pillow numpy matplotlib pyyaml pandas

## Step 1: Environment Setup & GPU Verification

In [ ]:
from pathlib import Path
import sys
import yaml
import torch
from ultralytics import YOLO

# 1. Locate repository root and add to sys.path
current_dir = Path.cwd()
repo_root = current_dir if (current_dir / "spark.yaml").exists() else current_dir.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.data.augment import SpaceImageAugmenter, run_augmentation, save_preview_grid

# 2. Check CUDA / GPU availability
device = 0 if torch.cuda.is_available() else "cpu"
device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"Using device: {device} ({device_name})")

# 3. Locate dataset config
spark_yaml = repo_root / "spark.yaml"
dataset_dir = repo_root / "data" / "processed" / "Swisscube"

with open(spark_yaml, "r") as f:
    cfg = yaml.safe_load(f) or {}

cfg["path"] = str(dataset_dir.resolve().as_posix())
cfg["train"] = "images/train"
cfg["val"] = "images/val"
cfg["test"] = "images/test"
cfg["names"] = {0: "swisscube"}

with open(spark_yaml, "w") as f:
    yaml.dump(cfg, f, sort_keys=False)

## Step 2: Photometric Augmentation Handoff (Noise, Blur, Exposure Jitter)
Wires `src/data/augment.py` to generate and verify photometric augmentations on the training split.

In [ ]:
img_dir = dataset_dir / "images" / "train"
existing_aug = list(img_dir.glob("aug1_*")) if img_dir.exists() else []

augmenter = SpaceImageAugmenter(
    brightness_range=(0.7, 1.3),
    contrast_range=(0.7, 1.3),
    blur_radius_range=(0.0, 2.0),
    noise_sigma_range=(2.0, 12.0),
    p_brightness_contrast=0.6,
    p_blur=0.5,
    p_noise=0.5,
    seed=42
)

# Generate preview grid
preview_path = dataset_dir / "augment_preview.png"
if dataset_dir.exists():
    save_preview_grid(dataset_dir, "train", augmenter, preview_path)

if len(existing_aug) == 0 and dataset_dir.exists():
    print("Wiring & applying photometric augmentation (sensor noise, Gaussian blur, brightness shift)...")
    n_imgs, n_lbls = run_augmentation(dataset_dir, "train", 1, augmenter)
    print(f"Photometric Augmentation complete: {n_imgs} augmented images added to training set.")
else:
    print(f"Photometric augmentation verified: {len(existing_aug)} augmented images present in training dataset.")

## Step 3: Launch Space-Domain YOLOv8 Baseline Training Loop
Runs 5-epoch training with GPU acceleration and space-domain hyperparameter augmentations.

In [ ]:
model = YOLO("yolov8n.pt")

results = model.train(
    data=str(spark_yaml.resolve().as_posix()),
    epochs=5,
    imgsz=1024,
    batch=8,
    cache=True,
    workers=4,
    amp=True,
    # Space domain hyperparameter augmentations
    degrees=180.0,   # Full 360-degree rotation invariance for tumbling space objects
    flipud=0.5,      # Vertical flip (zero-gravity environment)
    fliplr=0.5,      # Horizontal flip
    hsv_h=0.015,     # Color jitter
    hsv_s=0.7,       # Saturation jitter
    hsv_v=0.4,       # Brightness jitter
    erasing=0.4,     # Occlusion erasing for panel/antenna obscuration
    project=str((repo_root / "experiments").resolve().as_posix()),
    name="week1_smoke_test",
    device=device
)

print("Training Run Complete with Photometric & Orbital Augmentations!")

## Step 4: Evaluate Baseline Metrics
Inspect training metrics (loss curves, mAP50, mAP50-95).

In [ ]:
import pandas as pd

results_csv = repo_root / "experiments" / "week1_smoke_test" / "results.csv"
if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = [c.strip() for c in df.columns]
    print(df[['epoch', 'train/box_loss', 'train/cls_loss', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)']])